## COMP 341: Practical Machine Learning
## Homework assignment 2: movie critics
### Due: Wednesday, September 21 at 11:59pm on Gradescope

In this homework, we will be looking at movie reviews scraped from [Rotten Tomatoes](https://www.rottentomatoes.com). Rotten Tomatoes is a website that contains reviews for both movies and TV shows. Professional movie critics and any user on the site (audience) can write reviews and give movies a score from 0 (rotten / spilled) to 100 (fresh / upright). Often times the scores for a particular movie will diverge between critics and the audience, especially for polarizing movies or "cult classics." For this assignment, let's assume we are working for a movie production company that wants to find a critic for evaulating the viability of movie screenplays. In this scenario, we are tasked with finding a critic or critics that are in touch with what the general public likes and thus are a good predictor of how the audience will rate a movie.

While the scraped data is well structured, movie critics do not rate movies in a standardized way. This will be a good challenge for our data cleaning skills. We will also apply corrlelation and dimensionality reduction techniques to identify potential critic candidates.

As always, fill in missing code following `# TODO:` comments or `####### YOUR CODE HERE ########` blocks and be sure to answer the short answer questions marked with `[WRITE YOUR ANSWER HERE]` in the text.

All code in this notebook will be run sequentially so make sure things work in order! Be sure to also use good coding practices (e.g., logical variable names, comments as needed, etc), and make plots that are clear and legible.

For this assignment, there will be **15 points** allocated for general coding points:
* **10 points** for coding style
* **5 points** for code flow (accurate results when everything is run sequentially)

### Setup
First, we need to import some libraries that are necessary to complete the assignment.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *

Add additional modules/libraries to import here (rather than wherever you first use them below):

In [ ]:
# additional modules/libraries to import

The data for this assignment is hosted on [Google Drive](https://drive.google.com/drive/folders/1FKAjro4pdvoh-gKw1WPaig1zsWfXen0G?usp=sharing). There are two tables, critic reviews, and movies. Read both of these data sets in and get familiar with the structure.

We provide some code to get the files into your workspace below.

***If Google is rate-limiting the file downloads, you should follow the next set of instructions to copy the files into your on Google Drive and access the files that way instead.***

In [ ]:
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1op3hFWCn6onRlcBOKTliX3uHbbdus6Kx' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1op3hFWCn6onRlcBOKTliX3uHbbdus6Kx" -O rotten_tomatoes_critic_reviews.csv && rm -rf /tmp/cookies.txt
!wget --no-check-certificate 'https://drive.google.com/uc?export=download&id=1Jddf6szNHPlP0RtyTN06vQ5UDo4JAmko' -O 'rotten_tomatoes_movies.csv'

You only need to do the following once:
1. Go to 'My Drive' in your own Google Drive
2. Make a new folder named `comp341`
3. From the [Google Drive link]((https://drive.google.com/drive/folders/1FKAjro4pdvoh-gKw1WPaig1zsWfXen0G?usp=sharing)), click Download All. Unzip the resulting file and you will have a folder entitled `comp341-hw2` on your computer.
4. In the `comp341` folder you created in step 2, click `New -> Folder Upload` and select the `comp341-hw2` folder from your computer.

Now, we will mount your local Google Drive in colab so that you can read the files in.

In [ ]:
# note that this command will trigger a request from google to allow colab
# to access your files: you will need to accept the terms in order to access
# the files this way
from google.colab import drive
drive.mount('/content/drive')

# if you followed the instructions above exactly, the data files should be
# in comp341/comp341-hw2; if your files are in a different directory
# on your Google Drive, you will need to change the path below accordingly
DATADIR = '/content/drive/My Drive/comp341/comp341-hw2'

This command will check the contents of `DATADIR` to double check that you have access to the files. You can now use this file path to read in the data files for the assignment.

In [ ]:
!ls "$DATADIR"

### Part 0: Getting to Know the Data [11 pts]

In [ ]:
# TODO: read both tables into pandas

**Short Answer Question:** How many unique critics are there in this data?

`[WRITE YOUR ANSWER HERE]`

In [ ]:
# TODO: remove critics with no name

In [ ]:
# TODO: determine the number of critics

The number of movies a critic has reviewed has the possibility to vary greatly. If we want to analyze how well critics align with the average audience score we will need to only consider ones that have ranked many movies. Check the number of unique movie reviews per critic and look at the distribution across all critics.

In [ ]:
# TODO: how many movies does each critic score (not just categorize as
# fresh or rotten)?

In [ ]:
# TODO: plot the distribution of movies per critic

In [ ]:
# TODO: use the plot to select a sensible cutoff (number of movies reviewed)
# for critics to include in the analysis for Part 1

**Short Answer Question:** Give an explanation for the cutoff you chose above.

`[WRITE YOUR ANSWER HERE]`

### Part 1: Reviewer Bias [9 pts]

Some critics might tend to always say movies are terrible and some might just love movies too much and think all movies are great. Let's look at the fresh/rotten categories and check for trends per reviewer (using the filtered set of critics that you have selected from Part 0).

In [ ]:
# TODO: using review_type, determine the number of fresh vs rotten
# reviews per critic

In [ ]:
# TODO: plot the distribution of percentages of fresh reviews per critic

In [ ]:
# EXTRA CREDIT (1 pt): add color to the above plot to highlight which critics
# tend to write positive reviews (more than 50% fresh) or
# more negative reviews (more than 50% rotten)

**Short Answer Question:** Using the plot, would you say that critics are more likely to write a negative or positive movie review?

`[WRITE YOUR ANSWER HERE]`

### Part 2: Cleaning Scores [15 pts]

Critics do not grade movies on the same scale. To make them comparable, we will standardize the scale to 0-100 using the existing scores. Preserve the NaNs and focus on cleaning the different values for review_score.

For the subsequent analyses, we will be using a critic filter of at least 500 movies (which may be different from the cutoff and analyses you did in Part 1) so that everyone has a consistent analysis.

In [ ]:
# TODO: add a critic filter, keeping only critics that have
# reviewed at least 500 movies

In [ ]:
# TODO: look at the different values for review_type

In [ ]:
# TODO: convert all values in a reasonable, systematic manner to fall between
# 0 and 100 (be sure to comment on any design decisions)

In [ ]:
# TODO: show the that the values fall between 0 and 100 using describe()
# on the cleaned review_type field

### Part 3: Handling Missing Values & Dimensionality R [50 pts]
Unfortunately, critics do not review every movie. Even if we restrict critics to those that have reviewed at least 500 movies (as in part 2 above) there is likely only a very small set of total of movies that every critic has reviewed. This means there will be many missing values in the data. Fortunately, there are some tricks we can use to fill in the gaps. In this section we will try 3 different methods: 0, mean, KNN impute.



In [ ]:
# TODO: combine the movie and critic tables

In [ ]:
# TODO: remove movies without an audience score

In [ ]:
# TODO: fill in missing values with 0s

In [ ]:
# TODO: calculate the correlation between all critics and the audience

**Short Answer Question:** List the top 5 critic names that are the most correlated with the audience score.

`[WRITE YOUR ANSWER HERE]`

In [ ]:
# TODO: fill in missing values with the mean across movies
# (hint: be sure to exclude the audience score from the mean calculation!)

In [ ]:
# TODO: calculate the correlation between all critics and the audience

**Short Answer Question:** List the top 5 critic names that are the most correlated with the audience score.

`[WRITE YOUR ANSWER HERE]`

In [ ]:
# TODO: fill in missing values using KNN impute with K=5
# (hint: be sure to exclude the audience score from the imputation!)

In [ ]:
# TODO: calculate the correlation between all critics and the audience

**Short Answer Question:** List the top 5 critic names that are the most correlated with the audience score.

`[WRITE YOUR ANSWER HERE]`

**Short Answer Question:** Compare the top 5 critics identified using the 3 different imputation methods. Did you expect them to be the same or all different? What does this say about the choice of imputation method?

`[WRITE YOUR ANSWER HERE]`

In [ ]:
# TODO: run pca on the data where missing values have been filled in with 0s

In [ ]:
# TODO: plot pc1 vs pc2 coloring the audience score

In [ ]:
# TODO: run pca on the data with the KNN imputed values

In [ ]:
# TODO: plot pc1 vs pc2 coloring the audience score

**Short Answer Question:** The PCA plots look different depending on how NaNs are handled (with 0s or KNN). Does this imply that one missing value imputation method is better than the other?

`[WRITE YOUR ANSWER HERE]`

In [ ]:
# TODO: plot pc1 vs pc2 coloring the critics by whether or not they are biased
# to be positive or negative reviewers as identified in part 1

**Short Answer Question:** Based on your analysis, which 3 critics would you recommend for predicting the general audience score?

`[WRITE YOUR ANSWER HERE]`

### Bonus section (pick _one_ problem and solve for up to 10 points extra credit)

**Problem 1**: We searched for critics that are in touch with how the movie going audience might rate a movie and found varying results. Perhaps a better angle is to look at the publisher instead of individual reviewers. Using the best metrics you found for imputation, redo this analysis at the publisher level, being careful in the analysis (e.g., clearly indicating any design decisions for cutoffs).

OR

**Problem 2**: Analyze the PCA loadings from above made with data where missing values were filled in with KNN impute. Check if the important loadings have any relation to movie genre.

## To Submit
Download the notebook from Colab as a `.ipynb` notebook (`File > Download > Download .ipynb`) and upload it to the corresponding Gradescope assignment. Your assignment should be named `comp341-hw2.ipynb`.